<a href="https://colab.research.google.com/github/fael83/Smt4-Data-Mining/blob/main/Week7_Tugas2_Logistic_Regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving loan-test Tugas Minggu 7.csv to loan-test Tugas Minggu 7.csv


In [ ]:
data = pd.read_csv("loan-test Tugas Minggu 7.csv")
data.head()

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area
0,LP001015,Male,Yes,0,Graduate,No,5720,0,110.0,360.0,1.0,Urban
1,LP001022,Male,Yes,1,Graduate,No,3076,1500,126.0,360.0,1.0,Urban
2,LP001031,Male,Yes,2,Graduate,No,5000,1800,208.0,360.0,1.0,Urban
3,LP001035,Male,Yes,2,Graduate,No,2340,2546,100.0,360.0,NaN,Urban
4,LP001051,Male,No,0,Not Graduate,No,3276,0,78.0,360.0,1.0,Urban


In [ ]:
data.info()
data.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 367 entries, 0 to 366
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Loan_ID            367 non-null    object 
 1   Gender             356 non-null    object 
 2   Married            367 non-null    object 
 3   Dependents         357 non-null    object 
 4   Education          367 non-null    object 
 5   Self_Employed      344 non-null    object 
 6   ApplicantIncome    367 non-null    int64  
 7   CoapplicantIncome  367 non-null    int64  
 8   LoanAmount         362 non-null    float64
 9   Loan_Amount_Term   361 non-null    float64
 10  Credit_History     338 non-null    float64
 11  Property_Area      367 non-null    object 
dtypes: float64(3), int64(2), object(7)
memory usage: 34.5+ KB


,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History
count,367.000000,367.000000,362.000000,361.000000,338.000000
mean,4805.599455,1569.577657,136.132597,342.537396,0.825444
std,4910.685399,2334.232099,61.366652,65.156643,0.380150
min,0.000000,0.000000,28.000000,6.000000,0.000000
25%,2864.000000,0.000000,100.250000,360.000000,1.000000
50%,3786.000000,1025.000000,125.000000,360.000000,1.000000
75%,5060.000000,2430.500000,158.000000,360.000000,1.000000
max,72529.000000,24000.000000,550.000000,480.000000,1.000000


In [ ]:
data['Education'].value_counts()

,count
Education,
Graduate,283
Not Graduate,84


In [ ]:
x = data.drop('Education', axis=1)
y = data['Education']

In [ ]:
le = LabelEncoder()
y = le.fit_transform(y)

y[:10]

array([0, 0, 0, 0, 1, 1, 1, 1, 0, 1])

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(
    x, y,
    test_size=0.2,
    random_state=42
)

In [ ]:
print("Training data:", x_train.shape)
print("Testing data:", x_test.shape)

Training data: (293, 11)
Testing data: (74, 11)


In [ ]:
x_train_processed = x_train.copy()
x_test_processed = x_test.copy()

# 1. Drop 'Loan_ID' column as it is an identifier and not a feature
x_train_processed = x_train_processed.drop('Loan_ID', axis=1)
x_test_processed = x_test_processed.drop('Loan_ID', axis=1)

# 2. Handle missing numerical values using median imputation
# Identify numerical columns that might have NaNs
numerical_cols_with_nan = x_train_processed.select_dtypes(include=np.number).columns[x_train_processed.select_dtypes(include=np.number).isnull().any()].tolist()
for col in numerical_cols_with_nan:
    median_val = x_train_processed[col].median()
    x_train_processed[col].fillna(median_val, inplace=True)
    x_test_processed[col].fillna(median_val, inplace=True) # Use training median for test set

# 3. Handle missing categorical values and encode categorical features
# Identify categorical columns
categorical_cols = x_train_processed.select_dtypes(include='object').columns

# For 'Dependents' column, replace '3+' with '3' and convert to numeric
if 'Dependents' in categorical_cols:
    x_train_processed['Dependents'] = x_train_processed['Dependents'].replace('3+', '3').astype(float)
    x_test_processed['Dependents'] = x_test_processed['Dependents'].replace('3+', '3').astype(float)
    # After replacing '3+', it's numerical. Now, impute if any NaNs exist.
    if x_train_processed['Dependents'].isnull().any():
        median_val_dep = x_train_processed['Dependents'].median()
        x_train_processed['Dependents'].fillna(median_val_dep, inplace=True)
        x_test_processed['Dependents'].fillna(median_val_dep, inplace=True)
    categorical_cols = categorical_cols.drop('Dependents') # Remove Dependents from categorical for get_dummies

# For other categorical columns, fill missing with mode and then one-hot encode
for col in categorical_cols:
    if x_train_processed[col].isnull().any():
        mode_val = x_train_processed[col].mode()[0]
        x_train_processed[col].fillna(mode_val, inplace=True)
        x_test_processed[col].fillna(mode_val, inplace=True) # Use training mode for test set

# Apply one-hot encoding to remaining categorical columns
x_train_processed = pd.get_dummies(x_train_processed, columns=categorical_cols, drop_first=True, dtype=int)
x_test_processed = pd.get_dummies(x_test_processed, columns=categorical_cols, drop_first=True, dtype=int)

# Align columns - crucial for consistency between train and test sets after one-hot encoding
train_cols = x_train_processed.columns
test_cols = x_test_processed.columns

missing_in_test = set(train_cols) - set(test_cols)
for c in missing_in_test:
    x_test_processed[c] = 0

missing_in_train = set(test_cols) - set(train_cols)
for c in missing_in_train:
    x_train_processed[c] = 0

x_test_processed = x_test_processed[train_cols]

# 4. Scale numerical features
scaler = StandardScaler()
# Identify columns that are numerical after all preprocessing
numerical_features_for_scaling = x_train_processed.select_dtypes(include=np.number).columns
x_train_processed[numerical_features_for_scaling] = scaler.fit_transform(x_train_processed[numerical_features_for_scaling])
x_test_processed[numerical_features_for_scaling] = scaler.transform(x_test_processed[numerical_features_for_scaling])

# Initialize and fit the Logistic Regression model
# Increased max_iter for convergence, and added solver='liblinear' as it handles small datasets well and supports L1/L2 penalties
model = LogisticRegression(max_iter=1000, solver='liblinear')
model.fit(x_train_processed, y_train)

/tmp/ipykernel_11799/3157617878.py:13: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  x_train_processed[col].fillna(median_val, inplace=True)
/tmp/ipykernel_11799/3157617878.py:14: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=T

LogisticRegression(max_iter=1000, solver='liblinear')

In [17]:
y_pred = model.predict(x_test_processed)

In [18]:
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

Accuracy: 0.7567567567567568
